# COVID-19 Global Forecasting - Big Data Analysis

**Course:** Big Data  |  **Author:** Luís Coimbra

An academic project exploring COVID-19 trends and forecasting with Pandas, StatsForecast, and PySpark.

**Dataset:** COVID-19 Global Forecasting (Week 3), Kaggle competition data.

## Project Overview

Explore historical confirmed COVID-19 cases and fatalities by country and region, compare forecasting approaches, and generate 90-day projections. The dataset contains cumulative counts, so daily changes must be calculated before interpreting incidence. **Historical academic analysis only - not a current epidemiological forecast.**

> **Data access:** This notebook expects `train.csv` and `test.csv` in a `data/` folder one level above the notebook (`../data/`). The supplied competition CSV files are packaged in the accompanying `data/` directory.

> **Reproducibility:** The original academic code was retained where practical. Several forecasting and Spark sections have been corrected, but the notebook has not been fully executed in this environment; review outputs after running it with the packaged CSV files. Model results should be treated as illustrative until evaluated using a chronological holdout.

**Workflow:** EDA → StatsForecast baselines → residual inspection → PySpark SQL and machine-learning examples.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

In [ ]:
# Dataset files are expected in ../data/ (see the notebook introduction).
# Google Drive mounting is no longer required.


In [ ]:
treino = pd.read_csv("../data/train.csv")
teste = pd.read_csv("../data/test.csv")

## 1. Exploratory Data Analysis

In [ ]:
from IPython.display import display, HTML

treino_html = "<h2>Treino(Head)</h2>\n" + treino.head().to_html()

teste_html = "<h2>Teste(Head)</h2>\n" + teste.head().to_html()

html = "<table><tr><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(treino_html, teste_html)

display(HTML(html))

In [ ]:
from IPython.display import display, HTML

treino2_html = "<h2>Treino(Tail)</h2>\n" + treino.tail().to_html()

teste2_html = "<h2>Teste(Tail)</h2>\n" + teste.tail().to_html()

html = "<table><tr><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(treino2_html, teste2_html)

display(HTML(html))

The training dataset includes country, province/state, date, cumulative confirmed cases, and cumulative fatalities. The test dataset contains the locations and dates for which predictions are requested.

### 1.1 Dataset Structure and Quality

In [ ]:
treino.info()

In [ ]:
teste.info()

Some province/state values are missing. The original analysis removed this column for the Pandas exploration; this simplifies the data but loses regional detail. Country-level aggregation is used for comparisons.

In [ ]:
treino = treino.drop("Province_State", axis=1)

teste= teste.drop("Province_State",axis=1)

In [ ]:
#Training

print("Dataset dimensions")
print(treino.shape)
print("")
print("Column data types")
print(treino.dtypes)
print("")
print("Missing values")
print(treino.isnull().sum())
print("")
print("Descriptive statistics")
print(treino.describe())

In [ ]:

#Test

print("Dataset dimensions")
print(teste.shape)
print("")
print("Column data types")
print(teste.dtypes)
print("")
print("Missing values")
print(teste.isnull().sum())
print("")
print("Descriptive statistics")
print(teste.describe())

### 1.2 Number of Countries and Dates

In [ ]:
# Training dataset
num_countries = treino['Country_Region'].nunique()
num_days = treino['Date'].nunique()
print(f'Training data: {num_countries} countries and {num_days} days')


In [ ]:
# Test dataset
num_countries2 = teste['Country_Region'].nunique()
num_days2 = teste['Date'].nunique()
print(f'Test data: {num_countries2} countries and {num_days2} days')


### 1.3 Reported Cases and Fatalities

In [ ]:
latest = (treino.sort_values('Date').groupby(['Country_Region', 'Province_State'], dropna=False).tail(1) if 'Province_State' in treino.columns else treino.sort_values('Date').groupby('Country_Region').tail(1))
total_cases = latest['ConfirmedCases'].sum()
total_deaths = latest['Fatalities'].sum()
case_fatality_ratio = 100 * total_deaths / total_cases if total_cases else 0
print('Total confirmed cases (latest per location):', total_cases)
print('Total fatalities (latest per location):', total_deaths)
print(f'Observed case fatality ratio: {case_fatality_ratio:.2f}%')


### 1.4 Daily New Cases and Fatalities Over Time

In [ ]:
# Daily *new* cases are the difference between successive cumulative observations.
country_daily = treino.groupby(['Country_Region', 'Date'], as_index=False)[['ConfirmedCases', 'Fatalities']].sum().sort_values(['Country_Region', 'Date'])
country_daily['NewCases'] = country_daily.groupby('Country_Region')['ConfirmedCases'].diff().fillna(country_daily['ConfirmedCases']).clip(lower=0)
country_daily['NewFatalities'] = country_daily.groupby('Country_Region')['Fatalities'].diff().fillna(country_daily['Fatalities']).clip(lower=0)
daily_cases = country_daily.groupby('Date')[['NewCases']].sum()
display(daily_cases)


In [ ]:
# Daily new fatalities, rather than the sum of cumulative fatalities.
daily_deaths = country_daily.groupby('Date')[['NewFatalities']].sum()
display(daily_deaths)


In [ ]:
fig, axs = plt.subplots(2, figsize=(22, 10))
daily_cases.plot(y="NewCases", ax=axs[0], title="Daily new confirmed cases")
axs[0].ticklabel_format(style='plain', axis='y')
daily_deaths.plot(y="NewFatalities", ax=axs[1], title="Daily new fatalities")
axs[1].ticklabel_format(style='plain', axis='y')


### 1.5 Top 10 Countries by Cases and Fatalities

In [ ]:
# Latest cumulative totals by country (do not sum across dates)
top_countries = latest.groupby('Country_Region')['ConfirmedCases'].sum().nlargest(10)
display(top_countries)


In [ ]:
top_countries_fat = latest.groupby('Country_Region')['Fatalities'].sum().nlargest(10)
display(top_countries_fat)


In [ ]:
# Color dictionary para o primeiro chart
colors_dict_1 = {'US': 'red', 'China': 'yellow', 'Italy': 'green', 'Spain': 'orange', 'Germany': 'purple','France':'blue','Iran':'black','United Kingdom':'brown','Korea, South':'pink','Switzerland':'magenta'}

# Create uma lista de cores correspondente aos countries para o primeiro chart
colors_1 = [colors_dict_1.get(country, 'gray') for country in top_countries.index]

# Color dictionary para o segundo chart
colors_dict_2 = {'US': 'red', 'China': 'yellow', 'Italy': 'green', 'Spain': 'orange', 'Germany': 'purple','France':'blue','Iran':'black','United Kingdom':'brown','Korea, South':'pink','Switzerland':'magenta'}

# Create uma lista de cores correspondente aos countries para o segundo chart
colors_2 = [colors_dict_2.get(country, 'gray') for country in top_countries_fat.index]

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(33, 6))

# bar chart para os countries com mais casos confirmados

ax1.bar(top_countries.index, top_countries.values,color=colors_1)
ax1.ticklabel_format(style='plain', axis='y')
ax1.set_xlabel('Country')
ax1.set_ylabel('Number of cases confirmados')
ax1.set_title('Countries com mais casos confirmados')

# bar chart para os countries com mais mortes

ax2.bar(top_countries_fat.index, top_countries_fat.values,color=colors_2)
ax2.ticklabel_format(style='plain', axis='y')
ax2.set_xlabel('Country')
ax2.set_ylabel('Number of fatalities')
ax2.set_title('Countries com mais mortes')

plt.show()


**Interpretation:** The relative ranking of countries differs between confirmed cases and deaths. Because these are cumulative observations reported repeatedly over time, comparisons should use the latest available date per location rather than sums across dates.

In [ ]:
italy_data = latest[latest['Country_Region'] == 'Italy']
total_cases = italy_data['ConfirmedCases'].sum()
total_deaths = italy_data['Fatalities'].sum()
print('Italy confirmed cases:', total_cases)
print('Italy fatalities:', total_deaths)
print(f'Italy observed case fatality ratio: {100 * total_deaths / total_cases:.2f}%' if total_cases else 'No confirmed cases')


**Italy case study:** The observed ratio of reported fatalities to confirmed cases can be high in the historical snapshot. This is a *case fatality ratio*, not an individual probability of death. Differences in age structure, testing, reporting, and healthcare capacity can influence comparisons.

### 1.6 Countries with the Fewest Reported Cases and Fatalities

In [ ]:
top_countries3 = latest.groupby('Country_Region')['ConfirmedCases'].sum().loc[lambda x: x > 0].nsmallest(10).reset_index(name='Cases')
display(top_countries3)


In [ ]:
top_countries4 = latest.groupby('Country_Region')['Fatalities'].sum().loc[lambda x: x > 0].nsmallest(10).reset_index(name='Deaths')
display(top_countries4)


### 1.7 Cumulative Confirmed Case Trends

In [ ]:
# Ten countries with the most cases, plus Portugal for comparison.
list_countries = list(dict.fromkeys([*top_countries.index, 'Portugal']))


In [ ]:
#Agrupamento dos data por "Date" e "Country_Region" e depois é somado o número de casos confirmados por cada dia e país

time_series_cntr=treino.groupby(['Date','Country_Region'])['ConfirmedCases'].sum()\
                  .reset_index().set_index('Date')

#Create a new DataFrame que contém os countries especificados na lista

df_countries_tm = time_series_cntr[time_series_cntr['Country_Region'].isin(list_countries)]

plt.figure(figsize=(20,8))
ax = sns.lineplot(x=df_countries_tm.index,
                  y="ConfirmedCases",
                  hue='Country_Region',
                  data=df_countries_tm,palette='muted').set_title('Cumulative Line')              


plt.legend(loc=2,prop={'size':10})
plt.title('Cumulative confirmed case trends')
plt.xticks(rotation=90)

plt.show()

**Trend interpretation:** Countries show different trajectories in the historical period. Reporting changes, case definitions, and aggregation effects may also influence apparent spikes.

### 1.8 Country-Level Cumulative Case Table

In [ ]:
Confirm_pivot=pd.pivot_table(treino,index='Date',columns='Country_Region',
                             values='ConfirmedCases',aggfunc=np.sum)
Confirm_pivot

### 1.9 Daily New Cases in Selected Countries and Portugal

In [ ]:
plt.figure(figsize=(20,20))
colors=['r','b','g','y','orange','purple','m','hotpink','violet','darkgreen','navy','brown']
for i,country in enumerate(list_countries): # list_countries corresponde aos 10 paises com mais casos
    Confirm=Confirm_pivot[Confirm_pivot[country]>0][country].diff().fillna(0) # Criação de uma nova variável "Confirm" que contém a diferença dos casos por cada país. Estes casos são obtidos da tabela dinâmica anterior. A função diff() é usada 
    # para calcular a diferença entre os casos confirmados em days consecutivos e a fillna() para preencher valores ausentes com 0.
    Confirm=Confirm[Confirm>0]# retira valores 0 da variável confirm.
    
    plt.subplot(4,3,i+1)
    Confirm.plot(color=colors[i],label=country,markersize=12,lw=5)    
    plt.xticks()
    plt.legend(title='Country')
    plt.title('Number of cases diários em {}'.format(country.upper()))
plt.tight_layout()

**Interpretation:** Daily changes are calculated from cumulative counts. Negative corrections and missing reporting dates require care; these plots show positive reported changes only.

### 1.10 Daily New Fatalities in Selected Countries and Portugal

In [ ]:
mortalidade_pivot=pd.pivot_table(treino,index='Date',columns='Country_Region',
                             values='Fatalities',aggfunc=np.sum)
mortalidade_pivot

In [ ]:
plt.figure(figsize=(20,16))
for i,country in enumerate(list_countries):
    Fatal_diff=mortalidade_pivot[(mortalidade_pivot[country]>0)][country].diff().fillna(0)
    Fatal_diff=Fatal_diff[Fatal_diff>0]
    plt.subplot(3,4,i+1)
    Fatal_diff.plot(color=colors[i],label=country.upper(),lw=5)
    plt.xticks(rotation=60)
    plt.title('Daily new fatalities na {}'.format(country.upper()))
    plt.legend(title='Country')
plt.tight_layout()

### 1.11 Relationship Between Confirmed Cases and Fatalities

In [ ]:
cc=treino.groupby('Date')['ConfirmedCases'].sum()
ft=treino.groupby('Date')['Fatalities'].sum()

fig = go.Figure(data=go.Scatter(x=cc, y=ft, mode='lines+markers'))
# Edit the layout
fig.update_layout(title='Correlação entre ConfirmedCases e Fatalities mundialmente',
                   xaxis_title='Confirmed Cases ',
                   yaxis_title='Fatalities')
fig.show()

correlation_matrix = np.corrcoef(cc, ft)
correlation_xy = correlation_matrix[0,1]
r_squared = correlation_xy**2

print(r_squared)

Both cumulative series often increase over time. This association alone does not establish causation or provide a reliable fatality risk estimate.

### 1.12 EDA Summary

- COVID-19 trajectories vary across countries and dates.
- Cumulative counts should not be summed across reporting dates.
- Reporting practices and testing affect comparisons.
- The analysis provides historical context for the forecasting exercises.

## 2. Time Series Forecasting with Pandas and StatsForecast

In [ ]:
#https://towardsdatascience.com/global-covid-19-forecasting-with-linear-regression-and-arima-c154c163acc1

In [ ]:
# Convert column "Date" para DateTime e set it as the DataFrame index
treino['Date']=pd.to_datetime(treino['Date'])
treino.set_index('Date',inplace=True)
print(treino)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
treino.head()

In [ ]:
teste.head()

In [ ]:
teste.info()

### 2.1 Naive, Seasonal Naive, and AutoARIMA

**Naive:** predicts the previous observation. **Seasonal Naive:** repeats the value from the corresponding point in the previous season. **AutoARIMA:** searches for a suitable ARIMA specification. These are illustrative forecasting approaches, not validated epidemiological models.

### A. Forecasting Confirmed Cases

In [ ]:
# Dado que irá ser feito a forecast para o número de casos , remove the column Fatalities

treino = treino.drop("Fatalities", axis=1)

In [ ]:
   
# Install if needed: %pip install statsforecast
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive,AutoARIMA , AutoETS



**StatsForecast input:** Each series requires `unique_id` (series identifier), `ds` (date), and `y` (observed value). The data must contain one observation per country and date.

In [ ]:
Y_df = (treino.reset_index().groupby(['Country_Region', 'Date'], as_index=False)['ConfirmedCases'].sum().rename(columns={'Country_Region':'unique_id', 'Date':'ds', 'ConfirmedCases':'y'}))
Y_df['ds'] = pd.to_datetime(Y_df['ds'])
Y_df = Y_df.sort_values(['unique_id', 'ds'])
display(Y_df)


In [ ]:
StatsForecast.plot(Y_df)

In [ ]:
# Season length specifies the repeating period

model_list = [Naive(),
             SeasonalNaive(season_length=7),
             AutoARIMA(season_length=7, trace = True, alias = 'autoARIMA'),
            ]  

sf = StatsForecast(df=Y_df, models=model_list, freq='D')

In [ ]:
# h= 90 com freq"D" logo irá ser feito uma forecast para 90 days.
forecast_df = sf.forecast(h=90, fitted=True)

forecast_df

In [ ]:
display(forecast_df)

In [ ]:
sf.plot(Y_df, forecast_df)

In [ ]:
# Index reset

forecast_df = (forecast_df
        .reset_index())

display(forecast_df)

#### Forecasts for Selected Countries

In [ ]:
# Forecast para os 10 countries com mais casos de covid : China , US , Italy , Spain , Germany , France , Iran , UK , South Korea , Switzerland

Z_china= forecast_df[forecast_df['unique_id'] == 'China']
Z_us= forecast_df[forecast_df['unique_id'] == 'US']
Z_italy= forecast_df[forecast_df['unique_id'] == 'Italy']
Z_spain= forecast_df[forecast_df['unique_id'] == 'Spain']
Z_germany= forecast_df[forecast_df['unique_id'] == 'Germany']
Z_france= forecast_df[forecast_df['unique_id'] == 'France']
Z_iran= forecast_df[forecast_df['unique_id'] == 'Iran']
Z_uk= forecast_df[forecast_df['unique_id'] == 'United Kingdom']
Z_southkorea= forecast_df[forecast_df['unique_id'] == 'Korea, South']
Z_switzerland= forecast_df[forecast_df['unique_id'] == 'Switzerland']

# Portugal 

Z_portugal= forecast_df[forecast_df['unique_id'] == 'Portugal']



In [ ]:
china_html = "<h2>China</h2>\n" + Z_china.to_html()
us_html = "<h2>US</h2>\n" + Z_us.to_html()
italy_html = "<h2>Italy</h2>\n" + Z_italy.to_html()
spain_html = "<h2>Spain</h2>\n" + Z_spain.to_html()
germany_html = "<h2>Germany</h2>\n" + Z_germany.to_html()
france_html = "<h2>France</h2>\n" + Z_france.to_html()
iran_html = "<h2>Iran</h2>\n" + Z_iran.to_html()
uk_html = "<h2>UK</h2>\n" + Z_uk.to_html()
sk_html = "<h2>SouthKorea</h2>\n" + Z_southkorea.to_html()
swtz_html = "<h2>Switzerland</h2>\n" + Z_switzerland.to_html()

pt_html =  "<h2>Portugal</h2>\n" + Z_portugal.to_html()

In [ ]:
# China, US , Italy

html ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(china_html,us_html,italy_html)

# Spain, Germany , France

html2 ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(spain_html,germany_html,france_html)

# Iran, UK , SK

html3 ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(iran_html,uk_html,sk_html)

# Switzerland , Portugal

html4 = "<table><tr><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(swtz_html, pt_html)


#### China, United States, and Italy

In [ ]:
display(HTML(html))

In [ ]:
#CHINA
sf.plot(Y_df[Y_df['unique_id'].isin(Z_china['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_china['unique_id'].unique())])

In [ ]:
#US
sf.plot(Y_df[Y_df['unique_id'].isin(Z_us['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_us['unique_id'].unique())])

In [ ]:
#ITALY
sf.plot(Y_df[Y_df['unique_id'].isin(Z_italy['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_italy['unique_id'].unique())])

#### Spain, Germany, and France

In [ ]:
display(HTML(html2))

In [ ]:
#SPAIN
sf.plot(Y_df[Y_df['unique_id'].isin(Z_spain['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_spain['unique_id'].unique())])


In [ ]:
#GERMANY
sf.plot(Y_df[Y_df['unique_id'].isin(Z_germany['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_germany['unique_id'].unique())])

In [ ]:
#FRANCE
sf.plot(Y_df[Y_df['unique_id'].isin(Z_france['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_france['unique_id'].unique())])

#### Iran, United Kingdom, and South Korea

In [ ]:
display(HTML(html3))

In [ ]:
#IRAN
sf.plot(Y_df[Y_df['unique_id'].isin(Z_iran['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_iran['unique_id'].unique())])

In [ ]:
#UNITED KINGDOM
sf.plot(Y_df[Y_df['unique_id'].isin(Z_uk['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_uk['unique_id'].unique())])

In [ ]:
#SOUTH KOREA
sf.plot(Y_df[Y_df['unique_id'].isin(Z_southkorea['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_southkorea['unique_id'].unique())])

#### Switzerland and Portugal

In [ ]:
display(HTML(html4))

In [ ]:
#SWITZERLAND
sf.plot(Y_df[Y_df['unique_id'].isin(Z_switzerland['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_switzerland['unique_id'].unique())])

In [ ]:
#PORTUGAL
sf.plot(Y_df[Y_df['unique_id'].isin(Z_portugal['unique_id'].unique())], forecast_df[forecast_df['unique_id'].isin(Z_portugal['unique_id'].unique())])

In [ ]:
# Inspect fitted values

insample_df = sf.forecast_fitted_values()
insample_df



In [ ]:
sf.plot(Y_df, insample_df.reset_index() if 'unique_id' not in insample_df.columns else insample_df)


### Forecast Residuals

In [ ]:
# Residual calculation(erros)

# residuals é o valor entre o valor observado e valor dos methods

resid_df = (insample_df
            .assign(resid_naive = insample_df.y - insample_df.Naive,
                    resid_snaive = insample_df.y - insample_df.SeasonalNaive)   # criamos uma nova coluna com a diferença entre os valores observados e dos metodos
)
resid_df

In [ ]:
# Residuals are inspected numerically below.
display(resid_df[['resid_naive', 'resid_snaive']].describe())


In [ ]:
# Residual diagnostics - correlograma dos resíduos

from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid_df.resid_naive.dropna(), bartlett_confint=False, lags=90);
plot_acf(resid_df.resid_snaive.dropna(), bartlett_confint=False, lags=90);

#### Examples: China, United States, and Portugal

In [ ]:
# Index reset
residuos = (resid_df.reset_index())
display(residuos)

#### China - Residuals

In [ ]:
china_df = residuos[residuos['unique_id'] == 'China']
resid_df_china = china_df.assign(resid_naive = china_df.y - china_df.Naive,
                            resid_snaive = china_df.y - china_df.SeasonalNaive)

display(resid_df_china)

#### United States - Residuals

In [ ]:
us_df = residuos[residuos['unique_id'] == 'US']
resid_df_us = us_df.assign(resid_naive = us_df.y - us_df.Naive,
                            resid_snaive = us_df.y - us_df.SeasonalNaive)

display(resid_df_us)

#### Portugal - Residuals

In [ ]:
portugal_df = residuos[residuos['unique_id'] == 'Portugal']
resid_df = portugal_df.assign(resid_naive = portugal_df.y - portugal_df.Naive,
                              resid_snaive = portugal_df.y - portugal_df.SeasonalNaive)

display(resid_df)

### B. Forecasting Fatalities

In [ ]:
   
# Install if needed: %pip install statsforecast
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive,AutoARIMA , AutoETS

In [ ]:
# Load original training data for the fatalities forecast
treino2 = pd.read_csv('../data/train.csv')
teste2 = pd.read_csv('../data/test.csv')
display(treino2.head())


In [ ]:
# Dado que irá ser feito forecast para o número de mortes , remove the column ConfirmedCases
treino2= treino2.drop("ConfirmedCases", axis=1)

In [ ]:
display(treino2)

In [ ]:
Y_df2 = (treino2.groupby(['Country_Region', 'Date'], as_index=False)['Fatalities'].sum().rename(columns={'Country_Region':'unique_id', 'Date':'ds', 'Fatalities':'y'}))
Y_df2['ds'] = pd.to_datetime(Y_df2['ds'])
Y_df2 = Y_df2.sort_values(['unique_id', 'ds'])
display(Y_df2)


In [ ]:
StatsForecast.plot(Y_df2)

In [ ]:
# Season length specifies the repeating period

model_list = [Naive(),
             SeasonalNaive(season_length=7),
             AutoARIMA(season_length=7, trace = True, alias = 'autoARIMA'),
            ]  

sf2 = StatsForecast(df=Y_df2, models=model_list, freq='D')

In [ ]:
# h= 90 com freq"D" logo irá ser feito uma forecast para 90 days.
forecast_df2 = sf2.forecast(h=90, fitted=True)

forecast_df2

In [ ]:
display(forecast_df2)

In [ ]:
sf2.plot(Y_df2, forecast_df2)

In [ ]:
# Index reset

forecast_df2 = (forecast_df2
        .reset_index())

display(forecast_df2)

In [ ]:
# Forecast para os 10 countries com mais mortes de covid : China , US , Italy , Spain , Germany , France , Iran , UK , South Korea , Switzerland

Z_china2= forecast_df2[forecast_df2['unique_id'] == 'China']
Z_us2= forecast_df2[forecast_df2['unique_id'] == 'US']
Z_italy2= forecast_df2[forecast_df2['unique_id'] == 'Italy']
Z_spain2= forecast_df2[forecast_df2['unique_id'] == 'Spain']
Z_germany2= forecast_df2[forecast_df2['unique_id'] == 'Germany']
Z_france2= forecast_df2[forecast_df2['unique_id'] == 'France']
Z_iran2= forecast_df2[forecast_df2['unique_id'] == 'Iran']
Z_uk2= forecast_df2[forecast_df2['unique_id'] == 'United Kingdom']
Z_southkorea2= forecast_df2[forecast_df2['unique_id'] == 'Korea, South']
Z_switzerland2= forecast_df2[forecast_df2['unique_id'] == 'Switzerland']

# Portugal 

Z_portugal2= forecast_df2[forecast_df2['unique_id'] == 'Portugal']

In [ ]:
china_html2 = "<h2>China</h2>\n" + Z_china2.to_html()
us_html2 = "<h2>US</h2>\n" + Z_us2.to_html()
italy_html2 = "<h2>Italy</h2>\n" + Z_italy2.to_html()
spain_html2 = "<h2>Spain</h2>\n" + Z_spain2.to_html()
germany_html2 = "<h2>Germany</h2>\n" + Z_germany2.to_html()
france_html2 = "<h2>France</h2>\n" + Z_france2.to_html()
iran_html2 = "<h2>Iran</h2>\n" + Z_iran2.to_html()
uk_html2 = "<h2>UK</h2>\n" + Z_uk2.to_html()
sk_html2 = "<h2>SouthKorea</h2>\n" + Z_southkorea2.to_html()
swtz_html2 = "<h2>Switzerland</h2>\n" + Z_switzerland2.to_html()

pt_html2 =  "<h2>Portugal</h2>\n" + Z_portugal2.to_html()

In [ ]:
# China, US , Italy

html5 ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(china_html2,us_html2,italy_html2)

# Spain, Germany , France

html6 ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(spain_html2,germany_html2,france_html2)

# Iran, UK , SK

html7 ="<table><tr><td style='padding-right:60px'>{}</td><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(iran_html2,uk_html2,sk_html2)

# Switzerland , Portugal

html8 = "<table><tr><td style='padding-right:60px'>{}</td><td>{}</td></tr></table>".format(swtz_html2, pt_html2)


#### China, United States, and Italy

In [ ]:
display(HTML(html5))

#### Spain, Germany, and France

In [ ]:
display(HTML(html6))

#### Iran, United Kingdom, and South Korea

In [ ]:
display(HTML(html7))

#### Switzerland and Portugal

In [ ]:
display(HTML(html8))

In [ ]:
# Inspect fitted values

insample_df2 = sf2.forecast_fitted_values()
insample_df2

In [ ]:
# Compare the observed fatality series with the fitted model values.
sf2.plot(Y_df2, insample_df2.reset_index() if 'unique_id' not in insample_df2.columns else insample_df2)


### Fatality Forecast Residuals

In [ ]:
# Residual calculation(erros)

# residuals é o valor entre o valor observado e valor dos methods

resid_df2 = (insample_df2
            .assign(resid_naive = insample_df2.y - insample_df2.Naive,
                    resid_snaive = insample_df2.y - insample_df2.SeasonalNaive)   # criamos uma nova coluna com a diferença entre os valores observados e dos metodos
)
resid_df2

In [ ]:
display(resid_df2[['resid_naive', 'resid_snaive']].describe())


In [ ]:
# Residual diagnostics - correlograma dos resíduos

from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid_df2.resid_naive.dropna(), bartlett_confint=False, lags=8);
plot_acf(resid_df2.resid_snaive.dropna(), bartlett_confint=False, lags=8);

## 3. Distributed Analysis and Machine Learning with PySpark

In [ ]:
# Install PySpark in your environment if it is not already available.
# %pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, lit, lag, when, coalesce
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, DecisionTreeRegressor
spark = SparkSession.builder.appName('COVID19-BigData-Forecasting').getOrCreate()


### 3.1 PySpark Exploratory Analysis

#### Training Dataset

In [ ]:
from pyspark.sql import SparkSession

# Create a Spark session 
spark = SparkSession.builder.appName("Leitura de Dados").getOrCreate()


# Training

treino_spark = spark.read.format("csv").option("header", "true").load("../data/train.csv")

# Show the first rows of the DataFrame
treino_spark.show()

In [ ]:
treino_spark.printSchema()

# Dataset information ( first 10 rows)

print("Dataset information")
treino_spark.collect()[:10]

In [ ]:
#Métricas dos data
treino_spark.describe().show()

#### Test Dataset

In [ ]:
# Test

teste_spark = spark.read.format("csv").option("header", "true").load('../data/test.csv')
teste_spark.show()



In [ ]:
teste_spark.printSchema()

# Dataset information ( first 10 rows)
teste_spark.collect()[:10]

In [ ]:
teste_spark.describe().show()

### 3.2 Number of Countries and Dates

In [ ]:
# Training

#Countries
treino_spark.createOrReplaceTempView("treino_spark")
sqlDF = spark.sql('SELECT Country_Region FROM treino_spark GROUP BY Country_Region')
print("Total countries:",len(sqlDF.toPandas()['Country_Region']))

# Dias
sqlDF = spark.sql('Select Date from treino_spark Group By Date')
print("Total days:",len(sqlDF.toPandas()["Date"]))

In [ ]:
# Test

#Countries
teste_spark.createOrReplaceTempView("teste_spark")
sqlDF = spark.sql('SELECT Country_Region FROM teste_spark GROUP BY Country_Region')
print("Total countries:",len(sqlDF.toPandas()['Country_Region']))

# Dias
sqlDF = spark.sql('Select Date from teste_spark Group By Date')
print("Total days:",len(sqlDF.toPandas()["Date"]))


### 3.3 Top 10 Countries by Cases and Fatalities

In [ ]:
# Latest reported cumulative ConfirmedCases by country (not a sum across dates).
from pyspark.sql import functions as F
latest_spark = (treino_spark.withColumn('_row', F.row_number().over(Window.partitionBy('Country_Region', 'Province_State').orderBy(F.col('Date').desc())))
                .filter(F.col('_row') == 1))
country_totals = latest_spark.groupBy('Country_Region').agg(F.sum(F.col('ConfirmedCases').cast('double')).alias('ConfirmedCases'))
country_totals.orderBy(F.col('ConfirmedCases').desc()).show(10)


In [ ]:
# Latest reported cumulative Fatalities by country (not a sum across dates).
from pyspark.sql import functions as F
latest_spark = (treino_spark.withColumn('_row', F.row_number().over(Window.partitionBy('Country_Region', 'Province_State').orderBy(F.col('Date').desc())))
                .filter(F.col('_row') == 1))
country_totals = latest_spark.groupBy('Country_Region').agg(F.sum(F.col('Fatalities').cast('double')).alias('Fatalities'))
country_totals.orderBy(F.col('Fatalities').desc()).show(10)


### 3.4 Countries with the Fewest Cases and Fatalities

In [ ]:
# Latest reported cumulative ConfirmedCases by country (not a sum across dates).
from pyspark.sql import functions as F
latest_spark = (treino_spark.withColumn('_row', F.row_number().over(Window.partitionBy('Country_Region', 'Province_State').orderBy(F.col('Date').desc())))
                .filter(F.col('_row') == 1))
country_totals = latest_spark.groupBy('Country_Region').agg(F.sum(F.col('ConfirmedCases').cast('double')).alias('ConfirmedCases'))
country_totals.orderBy(F.col('ConfirmedCases').asc()).show(10)


In [ ]:
# Latest reported cumulative Fatalities by country (not a sum across dates).
from pyspark.sql import functions as F
latest_spark = (treino_spark.withColumn('_row', F.row_number().over(Window.partitionBy('Country_Region', 'Province_State').orderBy(F.col('Date').desc())))
                .filter(F.col('_row') == 1))
country_totals = latest_spark.groupBy('Country_Region').agg(F.sum(F.col('Fatalities').cast('double')).alias('Fatalities'))
country_totals.orderBy(F.col('Fatalities').asc()).show(10)


### 3.5 Previous-Observation (Naive) Baseline in PySpark

In [ ]:
from pyspark.sql.functions import unix_timestamp
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import to_timestamp, to_date
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, when
from pyspark.sql.functions import unix_timestamp



In [ ]:

train = spark.read.format('csv').options(header=True, inferSchema=True).load('../data/train.csv')
test = spark.read.format('csv').options(header=True, inferSchema=True).load('../data/test.csv')

# Estrutura
train.printSchema()
test.printSchema()

# Converting column "Date" to Date type
train = train.withColumn('Date', col('Date').cast('date'))
test = test.withColumn('Date', col('Date').cast('date'))


In [ ]:
test = test.withColumn('Forecast_Cases', lit(None).cast('double'))
test = test.withColumn('Forecast_Fatalities', lit(None).cast('double'))


In [ ]:
test.show()

In [ ]:
# Last observed cumulative counts per country/province as a simple baseline.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
location_window = Window.partitionBy('Country_Region', 'Province_State').orderBy(F.col('Date').desc())
latest_train = (train.withColumn('_rank', F.row_number().over(location_window))
                .filter(F.col('_rank') == 1)
                .select('Country_Region', 'Province_State',
                        F.col('ConfirmedCases').alias('Forecast_Cases'),
                        F.col('Fatalities').alias('Forecast_Fatalities')))
# Spark null keys do not match in a regular join: use null-safe equality.
t = test.alias('t')
l = latest_train.alias('l')
test = (t.join(l, (F.col('t.Country_Region') == F.col('l.Country_Region')) &
               F.col('t.Province_State').eqNullSafe(F.col('l.Province_State')), 'left')
        .select('t.*', 'l.Forecast_Cases', 'l.Forecast_Fatalities')
        .fillna({'Forecast_Cases': 0.0, 'Forecast_Fatalities': 0.0}))


In [ ]:
# Feature examples for training and test data.
lag_window = Window.partitionBy('Country_Region', 'Province_State').orderBy('Date')
train = (train.withColumn('ConfirmedCases_Prev', F.lag('ConfirmedCases').over(lag_window))
         .withColumn('Fatalities_Prev', F.lag('Fatalities').over(lag_window))
         .fillna({'ConfirmedCases_Prev': 0.0, 'Fatalities_Prev': 0.0}))
train_features = train.select('Country_Region', 'Province_State', 'Date', 'ConfirmedCases_Prev', 'Fatalities_Prev')
test_features = test.select('ForecastId', 'Country_Region', 'Province_State', 'Date', 'Forecast_Cases', 'Forecast_Fatalities')


In [ ]:
# Assemble the historical lag features for illustration.
assembler = VectorAssembler(inputCols=['ConfirmedCases_Prev', 'Fatalities_Prev'], outputCol='features')


In [ ]:
train_features = train_features.drop('features')


In [ ]:
train_features = assembler.transform(train_features)
# Test predictions are already available in test_features from the naive baseline.
display(test_features.limit(5).toPandas())


In [ ]:
# The naive forecast was computed above; no recursive null lag calculation is needed.


In [ ]:
# The naive forecast was computed above; no recursive null lag calculation is needed.


In [ ]:
# The naive forecast was computed above; no recursive null lag calculation is needed.


In [ ]:
train.show()

In [ ]:
test.show()


### 3.6 Linear Regression

Spark ML requires numeric features. Dates can be represented numerically, but a timestamp alone is a weak predictor and should not be interpreted as an epidemiological model.

In [ ]:
# Province_State was excluded in the original exploratory approach

#treino_spark=treino_spark.drop("Province_State")
#teste_spark=teste_spark.drop("Province_State")
#treino_spark.show()

In [ ]:
treino_spark = treino_spark.withColumn('Date', unix_timestamp('Date').cast('double'))
treino_spark.show(5)


In [ ]:
teste_spark = teste_spark.withColumn('Date', unix_timestamp('Date').cast('double'))
teste_spark.show(5)


In [ ]:
# Register the transformed Spark DataFrame before querying it.
treino_spark.createOrReplaceTempView('treino_spark')
N_NULL_DF = spark.sql('SELECT * FROM treino_spark WHERE Province_State IS NOT NULL')


In [ ]:
# Index province/state values present in the training data.
indexer = StringIndexer(inputCol='Province_State', outputCol='Province_StateIndex', handleInvalid='keep')
indexed = indexer.fit(N_NULL_DF).transform(N_NULL_DF)
indexed.show(5)


In [ ]:
# Index country names for the linear regression features.
country_indexer = StringIndexer(inputCol='Country_Region', outputCol='Country_RegionIndex', handleInvalid='keep')
indexed = country_indexer.fit(indexed).transform(indexed)
indexed.show(5)


In [ ]:
indexed.show()

In [ ]:
# Date is already numeric (Unix seconds) after the conversion above.


In [ ]:
from pyspark.sql.functions import col
#convert a coluna ConfirmedCases to numeric
indexed= indexed.withColumn("ConfirmedCases", col("ConfirmedCases").cast("float"))

In [ ]:
from pyspark.sql.functions import col

indexed = indexed.withColumn("Fatalities", col("Fatalities").cast("double"))


In [ ]:
# Keep the original features and train a linear regression model on fatalities.
assembler = VectorAssembler(inputCols=['Province_StateIndex', 'Country_RegionIndex', 'Date'], outputCol='features')
train_df = assembler.transform(indexed)
lr = LinearRegression(featuresCol='features', labelCol='Fatalities', maxIter=100)
lr_model = lr.fit(train_df)
print('Coefficients:', lr_model.coefficients)
print('Intercept:', lr_model.intercept)
trainingSummary = lr_model.summary
print('RMSE:', trainingSummary.rootMeanSquaredError)
print('MAE:', trainingSummary.meanAbsoluteError)
print('MSE:', trainingSummary.meanSquaredError)


In [ ]:
fatalities_lr_predictions = lr_model.transform(train_df)
fatalities_lr_predictions.select('Country_Region', 'Date', 'Fatalities', 'prediction').show(10)


### 3.7 Random Forest Regression

Random Forest is an ensemble of decision trees that can be used for regression. Here it provides an illustrative machine-learning baseline; robust temporal validation would be needed for a meaningful performance assessment.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor


In [ ]:
treino = spark.read.format("csv").option("header", "true").load("../data/train.csv")
teste = spark.read.format("csv").option("header", "true").load("../data/test.csv")

In [ ]:
from pyspark.sql.functions import unix_timestamp
treino = treino.withColumn('ConfirmedCases', col('ConfirmedCases').cast('double'))
treino = treino.withColumn('Fatalities', col('Fatalities').cast('double'))
treino = treino.withColumn('Date', unix_timestamp('Date').cast('double'))
teste = teste.withColumn('Date', unix_timestamp('Date').cast('double'))


In [ ]:
assembler = VectorAssembler(inputCols=["Date"], outputCol="features")
treino = assembler.transform(treino)
teste = assembler.transform(teste)


In [ ]:
rf = RandomForestRegressor(featuresCol='features', labelCol='ConfirmedCases', numTrees=10, maxDepth=5, seed=42)


In [ ]:
pipeline = Pipeline(stages=[rf])
model = pipeline.fit(treino)
rf_predictions = model.transform(teste).select('ForecastId', 'prediction')
rf_predictions.show(10)


### 3.8 Decision Tree Regression

A decision tree regressor partitions the feature space into regions and predicts a numerical target. Date-only features are retained here to stay close to the original academic exercise.

In [ ]:
treino_spark.describe()

In [ ]:
# Convert confirmedcases to numeric

from pyspark.sql.functions import col
treino_spark= treino_spark.withColumn("ConfirmedCases", col("ConfirmedCases").cast("float"))


In [ ]:
treino_spark.describe()

In [ ]:
train_data = treino_spark.select("Date", "ConfirmedCases")
test_data = teste_spark.select("Date", "ForecastId")


In [ ]:
from pyspark.ml.feature import VectorAssembler
# Preserve ForecastId when assembling test features for later output.
assembler = VectorAssembler(inputCols=['Date'], outputCol='features')
train_features = assembler.transform(train_data).select('features', 'ConfirmedCases')
test_features = assembler.transform(test_data).select('ForecastId', 'features')


In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

#Train a decision tree

dt = DecisionTreeRegressor(featuresCol="features", labelCol="ConfirmedCases")
model = dt.fit(train_features)


In [ ]:
# Keep ForecastId in the same DataFrame as the prediction to avoid an invalid join.
test_results = model.transform(test_features).select('ForecastId', 'prediction')
test_results.show(10)
